In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models, applications
import matplotlib.pyplot as plt
import numpy as np
import os
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')

print("✅ Mixed Precision Aktif! GPU akan berlari 2x lebih kencang.")

print("TensorFlow Version:", tf.__version__)
# Cek ketersediaan GPU
print("GPU Available:", tf.config.list_physical_devices('GPU'))

2026-03-11 04:37:00.960268: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773203821.141118      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773203821.186940      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773203821.592385      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773203821.592419      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773203821.592422      24 computation_placer.cc:177] computation placer alr

✅ Mixed Precision Aktif! GPU akan berlari 2x lebih kencang.
TensorFlow Version: 2.19.0
GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
# --- KONFIGURASI ---
TRAIN_DIR = '/kaggle/input/datasets/ananthu017/emotion-detection-fer/train' # Ganti dengan path folder train kamu
VALID_DIR = '/kaggle/input/datasets/ananthu017/emotion-detection-fer/test' # Ganti dengan path folder test/validasi
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
CLASSES = ['happy', 'sad', 'angry', 'surprised'] # Fokus ke 4 label game

# 1. Load Data Super Cepat (tanpa rescale 1./255!)
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    labels='inferred',
    label_mode='categorical',
    class_names=CLASSES,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    VALID_DIR,
    labels='inferred',
    label_mode='categorical',
    class_names=CLASSES,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# 2. Optimasi Pipeline (Prefetching agar GPU tidak nunggu)
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

print("✅ Data Pipeline Modern Siap!")

Found 19211 files belonging to 4 classes.


I0000 00:00:1773203857.098800      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Found 4810 files belonging to 4 classes.
✅ Data Pipeline Modern Siap!


In [3]:
# Ambil label murni untuk menghitung class weight
# tf.data agak unik cara ambil labelnya
train_labels = []
for images, labels in train_ds.unbatch():
    train_labels.append(np.argmax(labels.numpy()))

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)
class_weights_dict = dict(enumerate(class_weights))

print("⚖️ Bobot Kelas:")
for i, weight in class_weights_dict.items():
    print(f" - {CLASSES[i]}: {weight:.2f}")

⚖️ Bobot Kelas:
 - happy: 0.67
 - sad: 0.99
 - angry: 1.20
 - surprised: 1.51


In [4]:
# 1. Layer Augmentasi (Tetap sama)
data_augmentation = tf.keras.Sequential([
  layers.RandomFlip("horizontal"),
  layers.RandomRotation(0.1),
  layers.RandomZoom(0.1),
  layers.RandomTranslation(0.1, 0.1),
], name="gpu_augmentation")

# 2. LOAD CONVNEXT-TINY (Bintang Utamanya!)
base_model = applications.ConvNeXtTiny(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False # Freeze dulu untuk Phase 1

# 3. Rakit Arsitektur
inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = base_model(x, training=False)

# Transisi Classification Head (Sama dengan sebelumnya)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(len(CLASSES), activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)

# Gunakan Label Smoothing di compile agar tidak overconfident
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1), # <--- TWEAK PAMUNGKAS
    metrics=['accuracy']
)

model.summary()

111650432/111650432 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step


Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gpu_augmentation (Sequential)   │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ convnext_tiny (Functional)      │ (None, 7, 7, 768)      │    27,820,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 768)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 768)            │         3,072 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 768)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        98,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 27,922,660 (106.52 MB)

 Trainable params: 100,740 (393.52 KB)

 Non-trainable params: 27,821,920 (106.13 MB)

In [5]:
# Callback modern: Berhenti kalau nyangkut, turunkan LR kalau melambat
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.3, patience=2)
]

print("🚀 Memulai Pemanasan (Phase 1)...")
history_1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15, 
    callbacks=callbacks,
    class_weight=class_weights_dict # Paksa agar adil!
)

🚀 Memulai Pemanasan (Phase 1)...
Epoch 1/15


I0000 00:00:1773203915.624281      70 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1773203916.558009      70 service.cc:152] XLA service 0x7c317a3002d0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1773203916.558044      70 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1773203917.005820      70 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


601/601 ━━━━━━━━━━━━━━━━━━━━ 178s 252ms/step - accuracy: 0.4705 - loss: 1.4479 - val_accuracy: 0.6399 - val_loss: 0.9922 - learning_rate: 0.0010
Epoch 2/15
601/601 ━━━━━━━━━━━━━━━━━━━━ 107s 178ms/step - accuracy: 0.5851 - loss: 1.0764 - val_accuracy: 0.6543 - val_loss: 0.9659 - learning_rate: 0.0010
Epoch 3/15
601/601 ━━━━━━━━━━━━━━━━━━━━ 107s 178ms/step - accuracy: 0.6016 - loss: 1.0503 - val_accuracy: 0.6617 - val_loss: 0.9518 - learning_rate: 0.0010
Epoch 4/15
601/601 ━━━━━━━━━━━━━━━━━━━━ 107s 179ms/step - accuracy: 0.6115 - loss: 1.0282 - val_accuracy: 0.6657 - val_loss: 0.9383 - learning_rate: 0.0010
Epoch 5/15
601/601 ━━━━━━━━━━━━━━━━━━━━ 108s 180ms/step - accuracy: 0.6204 - loss: 1.0239 - val_accuracy: 0.6798 - val_loss: 0.9289 - learning_rate: 0.0010
Epoch 6/15
601/601 ━━━━━━━━━━━━━━━━━━━━ 108s 180ms/step - accuracy: 0.6237 - loss: 1.0222 - val_accuracy: 0.6736 - val_loss: 0.9332 - learning_rate: 0.0010
Epoch 7/15
601/601 ━━━━━━━━━━━━━━━━━━━━ 108s 180ms/step - accuracy: 0.6232 

In [6]:
print("🔓 Membuka SEMUA Gembok EfficientNet (Full Fine-Tuning Phase 2)...")

# 1. Unfreeze SELURUH layer EfficientNetB0
base_model.trainable = True

# Opsional tapi penting: Biarkan layer BatchNormalization tetap beku 
# agar tidak merusak statistik mean/variance dari ImageNet
for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

# 2. Compile ulang dengan Learning Rate yang SUPER KECIL (5e-5)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-5), # Turun dari 1e-4 ke 5e-5
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# 3. Callback lebih agresif untuk mencari akurasi tertinggi
callbacks_phase2 = [
    # Save bobot model TERBAIK saja, bukan bobot di akhir epoch
    tf.keras.callbacks.ModelCheckpoint(
        "best_efficientnet_game.keras", 
        monitor="val_accuracy", 
        save_best_only=True, 
        mode="max", 
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.2, patience=3, min_lr=1e-6)
]

print("🔥 Memulai Full Fine-Tuning...")
history_2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=25,  # Karena sudah pakai GPU, hajar saja 25 Epoch
    callbacks=callbacks_phase2,
    class_weight=class_weights_dict
)

print("🎉 Selesai! Cek file 'best_efficientnet_game.keras' untuk model dengan akurasi tertinggi.")

🔓 Membuka SEMUA Gembok EfficientNet (Full Fine-Tuning Phase 2)...
🔥 Memulai Full Fine-Tuning...
Epoch 1/25


2026-03-11 05:07:13.759633: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-11 05:07:13.954382: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-11 05:07:14.157391: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-11 05:07:14.333907: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


455/601 ━━━━━━━━━━━━━━━━━━━━ 1:22 563ms/step - accuracy: 0.6871 - loss: 0.7866

2026-03-11 05:12:06.948070: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-11 05:12:07.124370: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-11 05:12:07.360846: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-11 05:12:07.555478: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


601/601 ━━━━━━━━━━━━━━━━━━━━ 0s 627ms/step - accuracy: 0.6951 - loss: 0.7697
Epoch 1: val_accuracy improved from -inf to 0.76008, saving model to best_efficientnet_game.keras
601/601 ━━━━━━━━━━━━━━━━━━━━ 491s 706ms/step - accuracy: 0.6951 - loss: 0.7696 - val_accuracy: 0.7601 - val_loss: 0.6303 - learning_rate: 5.0000e-05
Epoch 2/25
601/601 ━━━━━━━━━━━━━━━━━━━━ 0s 564ms/step - accuracy: 0.7800 - loss: 0.5912
Epoch 2: val_accuracy improved from 0.76008 to 0.80769, saving model to best_efficientnet_game.keras
601/601 ━━━━━━━━━━━━━━━━━━━━ 362s 603ms/step - accuracy: 0.7800 - loss: 0.5912 - val_accuracy: 0.8077 - val_loss: 0.5104 - learning_rate: 5.0000e-05
Epoch 3/25
601/601 ━━━━━━━━━━━━━━━━━━━━ 0s 563ms/step - accuracy: 0.8059 - loss: 0.5117
Epoch 3: val_accuracy improved from 0.80769 to 0.81767, saving model to best_efficientnet_game.keras
601/601 ━━━━━━━━━━━━━━━━━━━━ 362s 602ms/step - accuracy: 0.8059 - loss: 0.5117 - val_accuracy: 0.8177 - val_loss: 0.4831 - learning_rate: 5.0000e-05
